# Tutorial 05: Adapt a Recipe for Your Own Task

Every RL training run in Tinker follows the same structure. To train on a new task, you change three things:

1. **Dataset** -- your problems or prompts
2. **Reward function** -- how you grade completions
3. **Hyperparameters** -- learning rate, group size, batch size

The training loop itself stays the same. This tutorial walks through each piece.

## Our example task: format compliance

We will train a model to follow specific formatting instructions. Each prompt asks the model to respond in a particular format (JSON, numbered list, or haiku). The reward is binary: 1.0 if the output matches the format constraint, 0.0 otherwise.

This is a good starter task because the reward function is simple, deterministic, and gives clear signal.

## Step 1: Define the dataset

Each problem is a `(prompt, format_type)` pair. The prompt tells the model what to write and in what format. We split into train and held-out sets.

In [1]:
TRAIN_PROBLEMS = [
    ("Describe the water cycle in JSON format with keys: process, steps, importance.", "json"),
    ("List 5 benefits of reading books. Use a numbered list.", "numbered_list"),
    ("Write a haiku about mountains.", "haiku"),
    ("Describe photosynthesis in JSON format with keys: inputs, process, outputs.", "json"),
    ("List 4 reasons to learn a new language. Use a numbered list.", "numbered_list"),
    ("Write a haiku about rain.", "haiku"),
    ("Describe gravity in JSON format with keys: definition, formula, examples.", "json"),
    ("List 3 tips for better sleep. Use a numbered list.", "numbered_list"),
    ("Write a haiku about the ocean.", "haiku"),
    ("Describe the solar system in JSON format with keys: star, planets, facts.", "json"),
    ("List 4 benefits of exercise. Use a numbered list.", "numbered_list"),
    ("Write a haiku about autumn leaves.", "haiku"),
]

EVAL_PROBLEMS = [
    (
        "Describe machine learning in JSON format with keys: definition, types, applications.",
        "json",
    ),
    ("List 3 ways to reduce waste. Use a numbered list.", "numbered_list"),
    ("Write a haiku about snow.", "haiku"),
]

print(f"Train: {len(TRAIN_PROBLEMS)} problems, Eval: {len(EVAL_PROBLEMS)} problems")

Train: 12 problems, Eval: 3 problems


## Step 2: Define the reward function

The reward function grades a model completion against the format constraint. Each checker is simple and deterministic -- this is intentional. A clear reward signal is essential for RL to work.

When writing your own reward function, test it on a few examples before training.

In [2]:
import json
import re


def check_json(text: str) -> bool:
    """Check if the response contains valid JSON."""
    # Try to find a JSON object in the text (possibly wrapped in markdown fences)
    match = re.search(r"\{[^{}]*\}", text, re.DOTALL)
    if match is None:
        return False
    try:
        json.loads(match.group())
        return True
    except json.JSONDecodeError:
        return False


def check_numbered_list(text: str) -> bool:
    """Check if the response contains a numbered list (at least 2 items)."""
    lines = [line.strip() for line in text.strip().splitlines() if line.strip()]
    numbered = [line for line in lines if re.match(r"^\d+[\.\)]\s", line)]
    return len(numbered) >= 2


def check_haiku(text: str) -> bool:
    """Check if the response is roughly 3 lines (haiku structure)."""
    lines = [line.strip() for line in text.strip().splitlines() if line.strip()]
    return len(lines) == 3


FORMAT_CHECKERS = {
    "json": check_json,
    "numbered_list": check_numbered_list,
    "haiku": check_haiku,
}


def get_reward(text: str, format_type: str) -> float:
    """Return 1.0 if the text matches the format, 0.0 otherwise."""
    checker = FORMAT_CHECKERS[format_type]
    return 1.0 if checker(text) else 0.0


# Quick sanity check
assert get_reward('{"key": "value"}', "json") == 1.0
assert get_reward("not json at all", "json") == 0.0
assert get_reward("1. First\n2. Second\n3. Third", "numbered_list") == 1.0
assert (
    get_reward("Gentle spring rain\nPatters on the window pane\nFlowers start to bloom", "haiku")
    == 1.0
)
print("Reward function sanity checks passed")

Reward function sanity checks passed


## Step 3: Choose hyperparameters

Three key hyperparameters for RL training:

- **Learning rate**: Use `get_lr(model_name)` for a calibrated starting point. This accounts for model size and LoRA scaling.
- **`group_size`**: Number of rollouts per problem. More rollouts give better advantage estimates. Start with 4-8.
- **`batch_size`**: Number of problems per training step. Start small (2-4) and scale up.

In [3]:
from tinker_cookbook.hyperparam_utils import get_lr

MODEL_NAME = "Qwen/Qwen3-30B-A3B"
LORA_RANK = 32
GROUP_SIZE = 4  # rollouts per problem
BATCH_SIZE = 4  # problems per training step
MAX_TOKENS = 300  # max response length
N_STEPS = 8  # training steps

learning_rate = get_lr(MODEL_NAME)
print(f"Model: {MODEL_NAME}")
print(f"Recommended LR: {learning_rate:.2e}")
print(f"Group size: {GROUP_SIZE}, Batch size: {BATCH_SIZE}")
print(f"Rollouts per step: {GROUP_SIZE * BATCH_SIZE}")

/Users/yujia/Repos/tinker-cookbook/.claude/worktrees/quirky-discovering-lightning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model: Qwen/Qwen3-30B-A3B
Recommended LR: 4.99e-04
Group size: 4, Batch size: 4
Rollouts per step: 16


## Step 4: Setup clients and renderer

We create a training client (LoRA), a renderer for the model's chat template, and sampling parameters. The renderer converts between messages and tokens.

In [4]:
import tinker
import torch
from tinker import TensorData

from tinker_cookbook.renderers import get_renderer, get_text_content

service_client = tinker.ServiceClient()
training_client = service_client.create_lora_training_client(base_model=MODEL_NAME, rank=LORA_RANK)
tokenizer = training_client.get_tokenizer()
renderer = get_renderer("qwen3", tokenizer)

sampling_params = tinker.SamplingParams(
    max_tokens=MAX_TOKENS,
    stop=renderer.get_stop_sequences(),
)
adam_params = tinker.AdamParams(learning_rate=learning_rate, beta1=0.9, beta2=0.95, eps=1e-8)

print("Training client ready")

Training client ready


## Step 5: The training loop

This is a standard GRPO-style loop. For each training step:

1. **Sample**: For each problem in the batch, generate `group_size` completions
2. **Reward**: Grade each completion with the reward function
3. **Compute advantages**: Center rewards within each group (GRPO)
4. **Train**: Build datums and call `forward_backward` + `optim_step`

The loop below is the same structure used by `recipes/rl_loop.py` -- only the dataset and reward function are different.

In [5]:
for step in range(N_STEPS):
    # Pick a batch of problems (cycle through the dataset)
    batch = [
        TRAIN_PROBLEMS[i % len(TRAIN_PROBLEMS)]
        for i in range(step * BATCH_SIZE, (step + 1) * BATCH_SIZE)
    ]

    # Create a sampling client from current weights
    sampling_client = training_client.save_weights_and_get_sampling_client()

    # --- Sample rollouts for each problem ---
    futures = []
    prompts = []
    for prompt_text, _format_type in batch:
        messages = [{"role": "user", "content": prompt_text}]
        model_input = renderer.build_generation_prompt(messages)
        future = sampling_client.sample(
            prompt=model_input, num_samples=GROUP_SIZE, sampling_params=sampling_params
        )
        futures.append(future)
        prompts.append(model_input)

    # --- Collect results and compute rewards ---
    datums = []
    all_rewards = []

    for (_prompt_text, format_type), future, prompt in zip(batch, futures, prompts):
        result = future.result()

        # Score each completion in the group
        rewards = []
        tokens_list = []
        logprobs_list = []
        for seq in result.sequences:
            parsed_msg, _ = renderer.parse_response(seq.tokens)
            content = get_text_content(parsed_msg)
            reward = get_reward(content, format_type)
            rewards.append(reward)
            tokens_list.append(seq.tokens)
            logprobs_list.append(seq.logprobs)

        all_rewards.extend(rewards)

        # GRPO: center rewards within the group
        mean_reward = sum(rewards) / len(rewards)
        advantages = [r - mean_reward for r in rewards]

        # Skip if all advantages are zero (all completions got the same reward)
        if all(a == 0.0 for a in advantages):
            continue

        # Build training datums
        ob_len = prompt.length - 1
        for tokens, logprobs, advantage in zip(tokens_list, logprobs_list, advantages):
            assert logprobs is not None
            model_input = prompt.append(tinker.EncodedTextChunk(tokens=tokens[:-1]))
            target_tokens = [0] * ob_len + tokens
            padded_logprobs = [0.0] * ob_len + logprobs
            padded_advantages = [0.0] * ob_len + [advantage] * (model_input.length - ob_len)
            datums.append(
                tinker.Datum(
                    model_input=model_input,
                    loss_fn_inputs={
                        "target_tokens": TensorData.from_torch(torch.tensor(target_tokens)),
                        "logprobs": TensorData.from_torch(torch.tensor(padded_logprobs)),
                        "advantages": TensorData.from_torch(torch.tensor(padded_advantages)),
                    },
                )
            )

    # --- Training step ---
    if datums:
        fwd_bwd_future = training_client.forward_backward(datums, loss_fn="importance_sampling")
        optim_future = training_client.optim_step(adam_params)
        fwd_bwd_future.result()
        optim_future.result()

    mean_reward = sum(all_rewards) / len(all_rewards) if all_rewards else 0.0
    print(f"Step {step}: mean_reward={mean_reward:.2f}, datums={len(datums)}")

Step 0: mean_reward=0.25, datums=0


Step 1: mean_reward=0.38, datums=8


Step 2: mean_reward=0.38, datums=4


Step 3: mean_reward=0.12, datums=4


Step 4: mean_reward=0.31, datums=8


Step 5: mean_reward=0.31, datums=4


Step 6: mean_reward=0.31, datums=4


Step 7: mean_reward=0.50, datums=0


## Evaluate on held-out prompts

We sample from the trained model on prompts it has not seen during training and check format compliance.

In [6]:
eval_client = training_client.save_weights_and_get_sampling_client(name="format-rl-final")

for prompt_text, format_type in EVAL_PROBLEMS:
    messages = [{"role": "user", "content": prompt_text}]
    model_input = renderer.build_generation_prompt(messages)
    result = eval_client.sample(
        prompt=model_input,
        num_samples=1,
        sampling_params=tinker.SamplingParams(
            max_tokens=MAX_TOKENS, temperature=0.5, stop=renderer.get_stop_sequences()
        ),
    ).result()

    parsed_msg, _ = renderer.parse_response(result.sequences[0].tokens)
    content = get_text_content(parsed_msg)
    reward = get_reward(content, format_type)
    status = "PASS" if reward == 1.0 else "FAIL"

    print(f"[{status}] Format: {format_type}")
    print(f"  Prompt: {prompt_text}")
    print(f"  Response: {content[:200]}...")
    print()

[FAIL] Format: json
  Prompt: Describe machine learning in JSON format with keys: definition, types, applications.
  Response: <think>
Okay, the user wants me to describe machine learning in JSON format with specific keys: definition, types, and applications. Let me start by recalling what machine learning is. It's a subset o...



[PASS] Format: numbered_list
  Prompt: List 3 ways to reduce waste. Use a numbered list.
  Response: 

1. **Recycle properly** by sorting materials like paper, glass, and plastics to ensure they are processed into new products.  
2. **Reduce single-use plastics** by opting for reusable items such as ...



[FAIL] Format: haiku
  Prompt: Write a haiku about snow.
  Response: <think>
Okay, the user wants a haiku about snow. Let me start by recalling the structure: 5-7-5 syllables. First line, 5 syllables. Maybe something about the falling snow. "Silent flakes descend" – th...



## Checklist for adapting any recipe

To bring your own task, you need to change exactly three things:

1. **Dataset**: Replace `TRAIN_PROBLEMS` with your own `(prompt, metadata)` pairs. The metadata is whatever your reward function needs to grade completions.
2. **Reward function**: Replace `get_reward()` with your own grading logic. It should return a float -- higher is better. Test it on examples before training.
3. **Hyperparameters**: Use `get_lr(model_name)` for the learning rate. Start with `group_size=4-8` and `batch_size=2-4`, then scale up.

Common mistakes to avoid:
- A reward function that always returns the same value (no learning signal)
- Mismatched renderer and model (e.g., using `llama3` renderer with a Qwen model)
- Setting `max_tokens` too low for the expected output length

## Going further

For more complex tasks, see the built-in recipes:

- **Math reasoning**: `tinker_cookbook/recipes/math_rl/` -- GSM8K and MATH with symbolic grading
- **Code generation**: `tinker_cookbook/recipes/code_rl/` -- DeepCoder with sandbox execution
- **Multi-turn RL**: `tinker_cookbook/recipes/multiplayer_rl/` -- agents that interact over multiple turns
- **DPO / RLHF**: `tinker_cookbook/recipes/preference/` -- preference-based training

For production training with logging, checkpointing, and evaluation, use the `Env` / `EnvGroupBuilder` / `RLDataset` abstractions from `tinker_cookbook.rl.types` together with the training loop in `tinker_cookbook.rl.train`. See [RL Environments](../docs/rl/rl-envs.mdx) for the full guide.